# P5 — Elexon Generation Data: Full Extraction & Spatial Split

Locked decisions: Elexon Insights Solution API only, no legacy BMRS/ElexonDataPortal; mixed GB fleet, all fuel types; location (`dictionary_id`) is the split key, not individual BMU; half-hourly, full history; raw CSV per BMU is source of truth, parquet is a disposable cache; empty BMUs detected generically via the manifest, not hand-excluded.

In [1]:
import requests
import pandas as pd
import numpy as np
from pathlib import Path
from io import StringIO
from datetime import date, datetime, timedelta
import time
import json

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)

RAW_DIR = Path("../data/raw/generation")
RAW_DIR.mkdir(parents=True, exist_ok=True)
REF_DIR = Path("../data/reference")
REF_DIR.mkdir(parents=True, exist_ok=True)
INTERIM_DIR = Path("../data/interim")
INTERIM_DIR.mkdir(parents=True, exist_ok=True)

OSUKED_RAW = "https://raw.githubusercontent.com/OSUKED/Power-Station-Dictionary/main"
ELEXON_API = "https://data.elexon.co.uk/bmrs/api/v1"

REQUEST_DELAY_S = 0.1  # be a good netizen
MANIFEST_PATH = INTERIM_DIR / "extraction_manifest.csv"

# Global chunk floor: deliberately earlier than the ~2019-02-01 floor the
# 6-site exploration sample suggested, so the full pull can confirm or
# correct that rather than assume it. See Step 6 for the confirmed figure.
EXTRACTION_START_YEAR = 2015
EXTRACTION_END = date.today() - timedelta(days=10)  # ~5 working day publication lag

## Step 1 — Resumable extraction pipeline design

Manifest (`data/interim/extraction_manifest.csv`): one row per `(bmu, chunk_start, chunk_end)`, one calendar-year chunk per BMU. Resumable (skips `success`, retries `failed`/`pending`), 429/5xx retried with backoff, 0.1s delay between requests.

In [2]:
# Fresh pull of the OSUKED reference tables (the complete BMU universe for this
# extraction comes from fuel_types, per the brief - not just the exploration sample)
osuked_files = {
    "fuel_types": "data/linked-datapackages/bmu-fuel-types/fuel_types.csv",
    "plant_locations": "data/linked-datapackages/plant-locations/plant-locations.csv",
    "dictionary_ids": "data/dictionary/ids.csv",
}

osuked = {}
for name, path in osuked_files.items():
    resp = requests.get(f"{OSUKED_RAW}/{path}", timeout=30)
    resp.raise_for_status()
    osuked[name] = pd.read_csv(StringIO(resp.text))

bmu_universe = sorted(osuked["fuel_types"]["ngc_bmu_id"].dropna().str.strip().unique())
print(f"Full BMU universe from OSUKED fuel_types: {len(bmu_universe)} distinct BMUs")

Full BMU universe from OSUKED fuel_types: 462 distinct BMUs


In [3]:
MANIFEST_COLUMNS = ["ngc_bmu_id", "chunk_start", "chunk_end", "status", "rows_returned", "attempted_at", "error"]


def year_chunks(start_year: int, end_date: date):
    """Yield (chunk_start, chunk_end) date strings, one per calendar year."""
    chunks = []
    for year in range(start_year, end_date.year + 1):
        chunk_start = date(year, 1, 1)
        chunk_end = date(year, 12, 31) if year < end_date.year else end_date
        if chunk_start > end_date:
            break
        chunks.append((chunk_start.isoformat(), chunk_end.isoformat()))
    return chunks


def load_or_init_manifest(bmus, start_year=EXTRACTION_START_YEAR, end_date=EXTRACTION_END):
    if MANIFEST_PATH.exists():
        manifest = pd.read_csv(MANIFEST_PATH, dtype={"ngc_bmu_id": str})
    else:
        manifest = pd.DataFrame(columns=MANIFEST_COLUMNS)

    existing_keys = set(zip(manifest["ngc_bmu_id"], manifest["chunk_start"], manifest["chunk_end"]))
    new_rows = []
    for bmu in bmus:
        for chunk_start, chunk_end in year_chunks(start_year, end_date):
            key = (bmu, chunk_start, chunk_end)
            if key not in existing_keys:
                new_rows.append({
                    "ngc_bmu_id": bmu, "chunk_start": chunk_start, "chunk_end": chunk_end,
                    "status": "pending", "rows_returned": pd.NA, "attempted_at": pd.NA, "error": pd.NA,
                })
    if new_rows:
        manifest = pd.concat([manifest, pd.DataFrame(new_rows)], ignore_index=True)
        manifest.to_csv(MANIFEST_PATH, index=False)
    return manifest


def save_manifest(manifest):
    manifest.to_csv(MANIFEST_PATH, index=False)

In [4]:
RETRYABLE_STATUS = {429, 500, 502, 503, 504}


def fetch_chunk(bmu, chunk_start, chunk_end, max_attempts=3):
    """Fetch one (bmu, year) chunk from the B1610 stream endpoint.
    Retries 429/5xx with backoff; other HTTP errors (e.g. 400/404) fail immediately.
    """
    params = {"from": chunk_start, "to": chunk_end, "bmUnit": bmu}
    last_exc = None
    for attempt in range(1, max_attempts + 1):
        try:
            r = requests.get(f"{ELEXON_API}/datasets/B1610/stream", params=params, timeout=60)
        except (requests.exceptions.ConnectionError, requests.exceptions.Timeout) as exc:
            last_exc = exc
            if attempt < max_attempts:
                time.sleep(2 ** attempt)  # 2s, 4s backoff
            continue

        if r.status_code in RETRYABLE_STATUS:
            last_exc = requests.exceptions.HTTPError(f"retryable status {r.status_code}")
            if attempt < max_attempts:
                time.sleep(2 ** attempt)
            continue

        r.raise_for_status()  # non-retryable 4xx raises immediately, no retry loop
        return r.json()
    raise last_exc


def append_bmu_csv(bmu, records):
    if not records:
        return
    df = pd.DataFrame(records)
    path = RAW_DIR / f"{bmu}.csv"
    header = not path.exists()
    df.to_csv(path, mode="a", header=header, index=False)

In [5]:
def run_extraction(manifest, progress_every=25):
    """Work through every pending/failed row in the manifest, in place.
    Prints a progress line with a rough ETA every `progress_every` chunks attempted.
    Safe to interrupt and re-run: already-`success` rows are untouched here.
    """
    todo_mask = manifest["status"].isin(["pending", "failed"])
    todo_idx = manifest.index[todo_mask].tolist()
    total_todo = len(todo_idx)
    start_time = time.monotonic()
    total_rows_this_run = 0

    print(f"{total_todo} chunks to attempt ({(~todo_mask).sum()} already succeeded previously)")

    for n, idx in enumerate(todo_idx, start=1):
        row = manifest.loc[idx]
        bmu, chunk_start, chunk_end = row["ngc_bmu_id"], row["chunk_start"], row["chunk_end"]
        try:
            records = fetch_chunk(bmu, chunk_start, chunk_end)
            append_bmu_csv(bmu, records)
            manifest.loc[idx, ["status", "rows_returned", "attempted_at", "error"]] = (
                "success", len(records), datetime.now().isoformat(timespec="seconds"), pd.NA,
            )
            total_rows_this_run += len(records)
        except Exception as exc:
            manifest.loc[idx, ["status", "rows_returned", "attempted_at", "error"]] = (
                "failed", pd.NA, datetime.now().isoformat(timespec="seconds"), str(exc)[:200],
            )
        time.sleep(REQUEST_DELAY_S)

        if n % progress_every == 0 or n == total_todo:
            save_manifest(manifest)  # flush periodically so progress is durable, not just in memory
            elapsed = time.monotonic() - start_time
            rate = n / elapsed  # chunks/sec
            remaining = total_todo - n
            eta_s = remaining / rate if rate > 0 else float("nan")
            eta_min = eta_s / 60
            done_bmus = manifest.loc[manifest["status"] == "success", "ngc_bmu_id"].nunique()
            print(
                f"[{n}/{total_todo}] {n/total_todo:.1%} | "
                f"{done_bmus}/{len(bmu_universe)} BMUs with >=1 success | "
                f"{total_rows_this_run:,} rows this run | "
                f"{rate:.2f} chunks/s | ETA ~{eta_min:.1f} min"
            )

    save_manifest(manifest)
    return manifest

## Step 2 — Kick off the full pull

Full BMU universe (not the exploration sample). Long-running, resumable via the manifest; progress line every 25 chunks with a rate-based ETA.

In [6]:
manifest = load_or_init_manifest(bmu_universe)
print(f"Manifest: {len(manifest)} total chunks, {(manifest['status'] == 'success').sum()} already successful")

manifest = run_extraction(manifest)

print()
print("Pull finished (or manifest fully attempted this run).")
print(manifest["status"].value_counts())

Manifest: 6006 total chunks, 5544 already successful
462 chunks to attempt (5544 already succeeded previously)


[25/462] 5.4% | 462/462 BMUs with >=1 success | 139,790 rows this run | 2.28 chunks/s | ETA ~3.2 min


[50/462] 10.8% | 462/462 BMUs with >=1 success | 309,535 rows this run | 2.45 chunks/s | ETA ~2.8 min


[75/462] 16.2% | 462/462 BMUs with >=1 success | 539,190 rows this run | 2.39 chunks/s | ETA ~2.7 min


[100/462] 21.6% | 462/462 BMUs with >=1 success | 688,965 rows this run | 2.51 chunks/s | ETA ~2.4 min


...

[375/462] 81.2% | 462/462 BMUs with >=1 success | 2,695,950 rows this run | 2.62 chunks/s | ETA ~0.6 min


[400/462] 86.6% | 462/462 BMUs with >=1 success | 2,885,665 rows this run | 2.62 chunks/s | ETA ~0.4 min


[425/462] 92.0% | 462/462 BMUs with >=1 success | 3,005,485 rows this run | 2.65 chunks/s | ETA ~0.2 min


[450/462] 97.4% | 462/462 BMUs with >=1 success | 3,145,275 rows this run | 2.65 chunks/s | ETA ~0.1 min


[462/462] 100.0% | 462/462 BMUs with >=1 success | 3,195,200 rows this run | 2.67 chunks/s | ETA ~0.0 min

Pull finished (or manifest fully attempted this run).
status
success    6006
Name: count, dtype: int64


## Step 3 — Aggregation by (location, fuel type)

Join path: `plant_locations -> dictionary_ids (exploded) -> fuel_types`. Aggregation unit is `(dictionary_id, fuel_type)`, not location alone — a multi-technology location gets one series per fuel type. `dictionary_id` stays the split key.

In [7]:
# Rebuild the join path from exploration
ids = osuked["dictionary_ids"][["dictionary_id", "name", "ngc_bmu_id"]].copy()
ids = ids.dropna(subset=["ngc_bmu_id"])
ids["ngc_bmu_id"] = ids["ngc_bmu_id"].str.split(",")
ids_exploded = ids.explode("ngc_bmu_id")
ids_exploded["ngc_bmu_id"] = ids_exploded["ngc_bmu_id"].str.strip()

site_bmu_map = (
    osuked["plant_locations"]
    .merge(ids_exploded, on="dictionary_id", how="inner")
    .merge(osuked["fuel_types"], on="ngc_bmu_id", how="inner")
)
# Only keep BMUs we actually attempted to pull data for
site_bmu_map = site_bmu_map[site_bmu_map["ngc_bmu_id"].isin(bmu_universe)]

print(f"site_bmu_map: {len(site_bmu_map)} BMU rows across {site_bmu_map['dictionary_id'].nunique()} sites")
site_bmu_map.head()

site_bmu_map: 403 BMU rows across 213 sites


,dictionary_id,longitude,latitude,name,ngc_bmu_id,fuel_type,comments
0,10000,-3.603516,57.480403,Rothes Bio-Plant CHP,MARK-1,BIOMASS,NaN
1,10000,-3.603516,57.480403,Rothes Bio-Plant CHP,MARK-2,BIOMASS,NaN
2,10001,-1.267570,51.623630,Didcot,DIDC01G,OCGT,NaN
3,10001,-1.267570,51.623630,Didcot,DIDC02G,OCGT,NaN
4,10001,-1.267570,51.623630,Didcot,DIDC03G,OCGT,NaN


**Data-cleaning fix**: one location labelled `Wind` (lower-case) normalised to `WIND` — case-inconsistency, not a distinct category.

In [8]:
n_before = (site_bmu_map["fuel_type"] == "Wind").sum()
site_bmu_map["fuel_type"] = site_bmu_map["fuel_type"].replace({"Wind": "WIND"})
n_after = (site_bmu_map["fuel_type"] == "Wind").sum()
print(f"Normalised {n_before} BMU row(s) from 'Wind' to 'WIND' ({n_after} remaining, should be 0)")

Normalised 1 BMU row(s) from 'Wind' to 'WIND' (0 remaining, should be 0)


In [9]:
# Under the revised (dictionary_id, fuel_type) aggregation, a location with more
# than one fuel type is no longer a problem to flag - it just becomes more than
# one fuel-series. Reported here for context, not as a data-quality concern.
fuel_types_per_location = site_bmu_map.groupby("dictionary_id")["fuel_type"].nunique()
multi_fuel_locations = fuel_types_per_location[fuel_types_per_location > 1]

print(f"{len(multi_fuel_locations)} location(s) have BMUs spanning more than one fuel type - "
      f"each now becomes its own fuel-series rather than being blended:")
display(
    site_bmu_map[site_bmu_map["dictionary_id"].isin(multi_fuel_locations.index)]
    .sort_values(["dictionary_id", "fuel_type"])[["dictionary_id", "name", "fuel_type"]]
    .drop_duplicates(["dictionary_id", "fuel_type"])
)

13 location(s) have BMUs spanning more than one fuel type - each now becomes its own fuel-series rather than being blended:


,dictionary_id,name,fuel_type
6,10001,Didcot,CCGT
2,10001,Didcot,OCGT
8,10002,Aberthaw B,COAL
11,10002,Aberthaw B,OCGT
18,10004,Drax,BIOMASS
22,10004,Drax,COAL
24,10004,Drax,OCGT
31,10006,Ferrybridge C,COAL
35,10006,Ferrybridge C,OCGT
37,10007,Fiddlers Ferry,COAL


In [10]:
SITE_GEN_DIR = INTERIM_DIR / "site_generation"
SITE_GEN_DIR.mkdir(parents=True, exist_ok=True)


def aggregate_site_fuel(dictionary_id, fuel_type, bmus, longitude, latitude):
    """Sum half-hourly generation across the BMUs sharing one (dictionary_id, fuel_type).
    Returns None if none of these BMUs have any raw data pulled yet."""
    frames = []
    for bmu in bmus:
        path = RAW_DIR / f"{bmu}.csv"
        if path.exists():
            df = pd.read_csv(path, usecols=["settlementDate", "settlementPeriod", "quantity"])
            frames.append(df)
    if not frames:
        return None
    combined = pd.concat(frames, ignore_index=True)
    # Guard against any duplicate rows from a chunk retried after a partial write
    combined = combined.drop_duplicates()
    series = (
        combined.groupby(["settlementDate", "settlementPeriod"], as_index=False)["quantity"]
        .sum()
        .assign(dictionary_id=dictionary_id, fuel_type=fuel_type, longitude=longitude, latitude=latitude)
    )
    return series


def build_all_site_fuel_series(site_bmu_map, save=True):
    summaries = []
    for (dictionary_id, fuel_type), group in site_bmu_map.groupby(["dictionary_id", "fuel_type"]):
        longitude, latitude = group["longitude"].iloc[0], group["latitude"].iloc[0]
        series = aggregate_site_fuel(dictionary_id, fuel_type, group["ngc_bmu_id"].tolist(), longitude, latitude)
        if series is None:
            continue
        site_fuel_id = f"{dictionary_id}_{fuel_type}"
        if save:
            series.to_csv(SITE_GEN_DIR / f"{site_fuel_id}.csv", index=False)
        summaries.append({
            "site_fuel_id": site_fuel_id,
            "dictionary_id": dictionary_id,  # location / split key - never split within this
            "name": group["name"].iloc[0],
            "longitude": longitude,
            "latitude": latitude,
            "fuel_type": fuel_type,
            "n_bmus": group["ngc_bmu_id"].nunique(),
            "n_half_hours": len(series),
            "earliest_settlement_date": series["settlementDate"].min(),
            "latest_settlement_date": series["settlementDate"].max(),
        })
    return pd.DataFrame(summaries)


site_fuel_summary = build_all_site_fuel_series(site_bmu_map)
n_possible = site_bmu_map.groupby(["dictionary_id", "fuel_type"]).ngroups
print(f"Fuel-series with at least some pulled data: {len(site_fuel_summary)} / {n_possible} possible (location, fuel type) combinations")
print(f"Distinct locations represented: {site_fuel_summary['dictionary_id'].nunique()}")
site_fuel_summary.head()

Fuel-series with at least some pulled data: 199 / 228 possible (location, fuel type) combinations
Distinct locations represented: 189


,site_fuel_id,dictionary_id,name,longitude,latitude,fuel_type,n_bmus,n_half_hours,earliest_settlement_date,latest_settlement_date
0,10000_BIOMASS,10000,Rothes Bio-Plant CHP,-3.603516,57.480403,BIOMASS,2,130856,2019-02-01,2026-07-28
1,10001_CCGT,10001,Didcot,-1.267570,51.623630,CCGT,2,130856,2019-02-01,2026-07-28
2,10001_OCGT,10001,Didcot,-1.267570,51.623630,OCGT,4,130856,2019-02-01,2026-07-28
3,10004_BIOMASS,10004,Drax,-0.996631,53.736634,BIOMASS,4,130856,2019-02-01,2026-07-28
4,10004_COAL,10004,Drax,-0.996631,53.736634,COAL,2,130856,2019-02-01,2026-07-28


**Observations**: 199/228 possible (location, fuel type) combinations have data, spanning 189 locations; 13 locations span more than one fuel type.

## Step 4 — Convex hull classification (per unique location)

Hull computed once per unique location (213, independent of data availability), then `hull_status` joined onto every fuel-series at that location. Boundary locations are always train; interior are holdout-eligible.

In [11]:
from scipy.spatial import ConvexHull

unique_locations = site_bmu_map[["dictionary_id", "longitude", "latitude"]].drop_duplicates().reset_index(drop=True)
coords = unique_locations[["longitude", "latitude"]].to_numpy()
hull = ConvexHull(coords)
hull_idx = set(hull.vertices)

unique_locations["hull_status"] = ["boundary" if i in hull_idx else "interior" for i in range(len(unique_locations))]
print(f"Hull computed over {len(unique_locations)} unique locations (all locations from the join, with or without data)")
print(unique_locations["hull_status"].value_counts())

# Join hull_status back onto every fuel-series at each location
site_fuel_summary = site_fuel_summary.merge(
    unique_locations[["dictionary_id", "hull_status"]], on="dictionary_id", how="left"
)

n_boundary_locations = (unique_locations["hull_status"] == "boundary").sum()
n_boundary_locations_with_data = site_fuel_summary.loc[site_fuel_summary["hull_status"] == "boundary", "dictionary_id"].nunique()
print(f"\n{n_boundary_locations} boundary locations overall; {n_boundary_locations_with_data} of those have >= 1 fuel-series with data")
print(f"{len(site_fuel_summary)} fuel-series now carry a hull_status")

Hull computed over 213 unique locations (all locations from the join, with or without data)
hull_status
interior    202
boundary     11
Name: count, dtype: int64

11 boundary locations overall; 10 of those have >= 1 fuel-series with data
199 fuel-series now carry a hull_status


**Observations**: 11 boundary locations out of 213 (10 with data), 202 interior (189 with data).

## Step 5 — Fuel-type coverage (revised granularity)

Coverage counted per fuel-series. Stratified fuel types (≥7 interior sites) confirmed: WIND, CCGT, NPSHYD, NUCLEAR, OCGT.

In [12]:
fuel_type_coverage = (
    site_fuel_summary.groupby(["fuel_type", "hull_status"]).size().unstack(fill_value=0)
)
for col in ["boundary", "interior"]:
    if col not in fuel_type_coverage.columns:
        fuel_type_coverage[col] = 0
fuel_type_coverage["total"] = fuel_type_coverage[["boundary", "interior"]].sum(axis=1)
fuel_type_coverage = fuel_type_coverage.sort_values("total", ascending=False)

fuel_type_coverage.to_csv(REF_DIR / "site_fuel_type_coverage.csv")

# OCGT confirmed as a 5th stratified fuel type this phase, on the back of the
# previous run's threshold-crossing flag (11 interior sites, well over the line)
CONFIRMED_STRATIFIED = {"WIND", "CCGT", "NPSHYD", "NUCLEAR", "OCGT"}
INTERIOR_THRESHOLD = 7
newly_qualifying = fuel_type_coverage[
    (fuel_type_coverage["interior"] >= INTERIOR_THRESHOLD) & (~fuel_type_coverage.index.isin(CONFIRMED_STRATIFIED))
]

print(f"Confirmed stratified fuel types (>= {INTERIOR_THRESHOLD} interior sites): {sorted(CONFIRMED_STRATIFIED)}")
if len(newly_qualifying):
    print(f"\nFuel type(s) newly crossing the >= {INTERIOR_THRESHOLD} interior-site threshold, not yet in the confirmed set:")
    display(newly_qualifying)
else:
    print(f"\nNo additional fuel type crosses the >= {INTERIOR_THRESHOLD} interior-site threshold beyond the confirmed set.")

fuel_type_coverage

Confirmed stratified fuel types (>= 7 interior sites): ['CCGT', 'NPSHYD', 'NUCLEAR', 'OCGT', 'WIND']

No additional fuel type crosses the >= 7 interior-site threshold beyond the confirmed set.


hull_status,boundary,interior,total
fuel_type,,,
WIND,8,104,112
CCGT,0,38,38
NPSHYD,0,13,13
OCGT,1,11,12
NUCLEAR,1,7,8
COAL,0,5,5
BIOMASS,0,4,4
PS,0,4,4
RECIPROCATING,0,3,3


**Observations**: OCGT confirmed as the 5th stratified type (11 interior sites). Full table: WIND 112 (104 interior), CCGT 38, NPSHYD 13, OCGT 12 (11), NUCLEAR 8 (7), COAL 5, BIOMASS 4, PS 4, RECIPROCATING 3.

## Step 6 — History-length distribution (revised granularity) and candidate cutoffs

Re-checks whether the ~2019-02-01 floor still holds at this granularity. No cutoff applied — candidates proposed for Simon to pick from.

In [13]:
site_fuel_summary["earliest_settlement_date"] = pd.to_datetime(site_fuel_summary["earliest_settlement_date"])
site_fuel_summary["latest_settlement_date"] = pd.to_datetime(site_fuel_summary["latest_settlement_date"])

history_stats = site_fuel_summary["earliest_settlement_date"].describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9])
print("Per-fuel-series earliest available settlement date, distribution:")
print(history_stats)

print(f"\nEarliest start date across all fuel-series: {site_fuel_summary['earliest_settlement_date'].min().date()}")
print(f"Latest start date (shortest-history series, by start): {site_fuel_summary['earliest_settlement_date'].max().date()}")
print(f"Median start date: {site_fuel_summary['earliest_settlement_date'].median().date()}")

near_floor = site_fuel_summary["earliest_settlement_date"].between("2019-01-25", "2019-02-05")
print(f"\nFuel-series starting within +/-5 days of 2019-02-01: {near_floor.sum()} / {len(site_fuel_summary)} ({near_floor.mean():.1%})")

site_fuel_summary.groupby("fuel_type")["earliest_settlement_date"].agg(["min", "median", "max", "count"])

Per-fuel-series earliest available settlement date, distribution:
count                           199
mean     2019-03-10 17:29:14.773869
min             2019-02-01 00:00:00
10%             2019-02-01 00:00:00
25%             2019-02-01 00:00:00
50%             2019-02-01 00:00:00
75%             2019-02-01 00:00:00
90%             2019-02-01 00:00:00
max             2022-02-16 00:00:00
Name: earliest_settlement_date, dtype: object

Earliest start date across all fuel-series: 2019-02-01
Latest start date (shortest-history series, by start): 2022-02-16
Median start date: 2019-02-01

Fuel-series starting within +/-5 days of 2019-02-01: 189 / 199 (95.0%)


,min,median,max,count
fuel_type,,,,
BIOMASS,2019-02-01,2019-02-01,2019-02-01,4
CCGT,2019-02-01,2019-02-01,2019-02-01,38
COAL,2019-02-01,2019-02-01,2019-02-01,5
NPSHYD,2019-02-01,2019-02-01,2019-02-01,13
NUCLEAR,2019-02-01,2019-02-01,2019-02-01,8
OCGT,2019-02-01,2019-02-01,2019-02-01,12
PS,2019-02-01,2019-02-01,2019-02-01,4
RECIPROCATING,2019-02-01,2019-02-01,2019-02-01,3
WIND,2019-02-01,2019-02-01,2022-02-16,112


In [14]:
# Caveat check: EXTRACTION_START_YEAR (2015) bounds what we could possibly observe.
# If a meaningful number of series' earliest date sits right at that floor, it's a
# signal the true floor may be earlier still and the search window should be
# extended (the manifest makes that a cheap re-run, not a rebuild).
at_search_floor = site_fuel_summary["earliest_settlement_date"].between(f"{EXTRACTION_START_YEAR}-01-01", f"{EXTRACTION_START_YEAR}-01-07")
print(f"Fuel-series whose earliest date is within the first week of the {EXTRACTION_START_YEAR} search floor: {at_search_floor.sum()} / {len(site_fuel_summary)}")
if at_search_floor.sum() > 0:
    print("-> non-trivial: the true history floor for these series may predate our search window. Consider lowering EXTRACTION_START_YEAR and re-running (manifest will only fetch the newly-added earlier chunks).")
else:
    print("-> none/negligible: the search floor does not appear to be truncating real history.")

Fuel-series whose earliest date is within the first week of the 2015 search floor: 0 / 199
-> none/negligible: the search floor does not appear to be truncating real history.


### Candidate minimum-history cutoffs

History length = `latest - earliest` settlement date, not just start date. Three round-number candidates (~1/2/3 years) proposed, none applied.

In [15]:
site_fuel_summary["history_days"] = (
    site_fuel_summary["latest_settlement_date"] - site_fuel_summary["earliest_settlement_date"]
).dt.days

CANDIDATE_CUTOFFS_DAYS = [365, 730, 1095]  # ~1, ~2, ~3 years
total_series = len(site_fuel_summary)
total_locations = site_fuel_summary["dictionary_id"].nunique()

print(f"Baseline: {total_series} fuel-series across {total_locations} locations, no cutoff applied\n")
print("Candidate cutoffs (not applied - for Simon to choose):\n")
for cutoff in CANDIDATE_CUTOFFS_DAYS:
    below = site_fuel_summary["history_days"] < cutoff
    series_lost = below.sum()
    locations_after = site_fuel_summary.loc[~below, "dictionary_id"].nunique()
    locations_lost_entirely = total_locations - locations_after
    years = cutoff / 365
    print(
        f"- >= {cutoff} days (~{years:.0f} year{'s' if years != 1 else ''}): "
        f"{series_lost}/{total_series} fuel-series excluded, "
        f"{locations_lost_entirely}/{total_locations} locations would lose ALL their series"
    )

print("\nShortest 15 fuel-series by history length:")
site_fuel_summary[["site_fuel_id", "dictionary_id", "fuel_type", "history_days"]].sort_values("history_days").head(15)

Baseline: 199 fuel-series across 189 locations, no cutoff applied

Candidate cutoffs (not applied - for Simon to choose):

- >= 365 days (~1 year): 0/199 fuel-series excluded, 0/189 locations would lose ALL their series
- >= 730 days (~2 years): 0/199 fuel-series excluded, 0/189 locations would lose ALL their series
- >= 1095 days (~3 years): 3/199 fuel-series excluded, 2/189 locations would lose ALL their series

Shortest 15 fuel-series by history length:


,site_fuel_id,dictionary_id,fuel_type,history_days
6,10007_COAL,10007,COAL,972
7,10007_OCGT,10007,OCGT,972
79,10134_NUCLEAR,10134,NUCLEAR,1071
17,10021_CCGT,10021,CCGT,1330
109,10173_WIND,10173,WIND,1623
180,10291_WIND,10291,WIND,1623
11,10012_COAL,10012,COAL,1771
196,10308_WIND,10308,WIND,1771
12,10012_OCGT,10012,OCGT,1771
16,10014_OCGT,10014,OCGT,1794


**Observations**: floor still holds (189/199 series within 5 days of 2019-02-01). Candidate cutoffs are cheap here — ~1yr and ~2yr lose nothing; ~3yr excludes 3 series / 2 locations.

## Step 7 — Finalise the spatial split

Confirmed: ~2yr minimum-history cutoff pipeline-wide; stratified types (WIND, CCGT, NPSHYD, NUCLEAR, OCGT) split 70/15/15 with a floor of 1 in val/test; boundary and sub-threshold-interior locations always train; `dictionary_id` remains the split key throughout.

**A location is atomic — anything that pulls it in wins outright.** Two edge cases resolved on that principle: (1) 4 locations carry two stratified fuel types (CCGT+OCGT) — given their own combined split first, then each type's remaining quota filled from single-fuel locations; (2) 5 locations pair a stratified type with a sub-threshold one — pulled out of stratification entirely and forced to train, even if that shrinks the eligible pool for their stratified type.

In [16]:
MIN_HISTORY_DAYS = 730  # ~2 years, confirmed

below_cutoff = site_fuel_summary["history_days"] < MIN_HISTORY_DAYS
n_series_excluded = below_cutoff.sum()
locations_before = site_fuel_summary["dictionary_id"].nunique()
locations_after = site_fuel_summary.loc[~below_cutoff, "dictionary_id"].nunique()
locations_lost_entirely = locations_before - locations_after

print(f"Applying the confirmed {MIN_HISTORY_DAYS}-day (~2 year) minimum-history cutoff:")
print(f"  {n_series_excluded} / {len(site_fuel_summary)} fuel-series excluded")
print(f"  {locations_lost_entirely} / {locations_before} locations lose ALL their series and drop out entirely")

if n_series_excluded > 0:
    print("\nExcluded series:")
    display(site_fuel_summary.loc[below_cutoff, ["site_fuel_id", "dictionary_id", "fuel_type", "history_days"]])

EXPECTED_EXCLUDED_AT_2YR = 0  # per the earlier candidate-cutoff report (Step 6)
if n_series_excluded > EXPECTED_EXCLUDED_AT_2YR:
    print(f"\n*** STOP: {n_series_excluded} series excluded at the 2-year cutoff, more than the "
          f"{EXPECTED_EXCLUDED_AT_2YR} the earlier candidate-cutoff report suggested. Flag before treating the split as final. ***")
else:
    print(f"\nMatches expectation from the earlier candidate-cutoff report ({EXPECTED_EXCLUDED_AT_2YR} excluded at 2 years).")

filtered_summary = site_fuel_summary.loc[~below_cutoff].reset_index(drop=True)

Applying the confirmed 730-day (~2 year) minimum-history cutoff:
  0 / 199 fuel-series excluded
  0 / 189 locations lose ALL their series and drop out entirely

Matches expectation from the earlier candidate-cutoff report (0 excluded at 2 years).


### Build the split assignment

Interior locations grouped by which stratified fuel types they touch; multi-membership groups split first, single-fuel-type locations fill each type's remaining quota. Fixed random seed for reproducibility.

In [17]:
STRATIFIED_FUEL_TYPES = ["WIND", "CCGT", "NPSHYD", "NUCLEAR", "OCGT"]
VAL_FRACTION = 0.15
TEST_FRACTION = 0.15
SPLIT_RNG_SEED = 42


def stratified_counts(n):
    """val/test get round(n * fraction) with a hard floor of 1; train gets the remainder."""
    val_n = max(1, round(n * VAL_FRACTION))
    test_n = max(1, round(n * TEST_FRACTION))
    train_n = n - val_n - test_n
    return train_n, val_n, test_n


rng = np.random.default_rng(SPLIT_RNG_SEED)

interior = filtered_summary[filtered_summary["hull_status"].eq("interior")]

# A location carrying ANY sub-threshold fuel type is forced entirely to train -
# a location is atomic (in or out of the training set), and a train-only series
# pulls the whole location in with it. These locations never enter the
# stratification pool for their other (stratified) fuel types at all.
sub_threshold_locations = set(
    interior.loc[~interior["fuel_type"].isin(STRATIFIED_FUEL_TYPES), "dictionary_id"]
)
print(f"{len(sub_threshold_locations)} interior location(s) forced to train (carry a sub-threshold fuel type): "
      f"{sorted(sub_threshold_locations)}")

interior_strat_all = interior[interior["fuel_type"].isin(STRATIFIED_FUEL_TYPES)]

# Target train/val/test counts per fuel type, based on ALL interior locations with
# that fuel type (forced-train ones included) - the "true" population to split.
fuel_type_targets = {}
for ft in STRATIFIED_FUEL_TYPES:
    n = (interior_strat_all["fuel_type"] == ft).sum()
    train_n, val_n, test_n = stratified_counts(n)
    fuel_type_targets[ft] = {"n": n, "train": train_n, "val": val_n, "test": test_n}

print("\nPer-fuel-type targets (locations, not series - includes forced-train locations in n):")
for ft, t in fuel_type_targets.items():
    print(f"  {ft:8s} n={t['n']:3d} -> train={t['train']}, val={t['val']}, test={t['test']}")

# Eligible pool for actual val/test assignment: stratified fuel types only, and
# excludes any location forced to train by a co-located sub-threshold fuel type.
interior_strat_eligible = interior_strat_all[~interior_strat_all["dictionary_id"].isin(sub_threshold_locations)]
loc_fuel_sets = interior_strat_eligible.groupby("dictionary_id")["fuel_type"].apply(lambda s: frozenset(s))

# Group eligible locations by their exact stratified-fuel-type signature; multi-membership groups handled first
signature_groups = {}
for dictionary_id, fuel_set in loc_fuel_sets.items():
    signature_groups.setdefault(fuel_set, []).append(dictionary_id)

location_split = {}
already_assigned = {ft: {"train": 0, "val": 0, "test": 0} for ft in STRATIFIED_FUEL_TYPES}

multi_signatures = [fs for fs in signature_groups if len(fs) > 1]
single_signatures = [fs for fs in signature_groups if len(fs) == 1]

print(f"\n{len(multi_signatures)} multi-stratified-fuel-type location group(s) among the eligible pool: {[sorted(fs) for fs in multi_signatures]}")

for fuel_set in multi_signatures:
    locs = list(rng.permutation(signature_groups[fuel_set]))
    train_n, val_n, test_n = stratified_counts(len(locs))
    assigned = ["val"] * val_n + ["test"] * test_n + ["train"] * train_n
    for loc, split in zip(locs, assigned):
        location_split[loc] = split
        for ft in fuel_set:
            already_assigned[ft][split] += 1

for fuel_set in single_signatures:
    ft = next(iter(fuel_set))
    locs = list(rng.permutation(signature_groups[fuel_set]))
    target = fuel_type_targets[ft]
    remaining_val = min(max(0, target["val"] - already_assigned[ft]["val"]), len(locs))
    remaining_test = min(max(0, target["test"] - already_assigned[ft]["test"]), len(locs) - remaining_val)
    n_train = len(locs) - remaining_val - remaining_test
    assigned = ["val"] * remaining_val + ["test"] * remaining_test + ["train"] * n_train
    for loc, split in zip(locs, assigned):
        location_split[loc] = split
        already_assigned[ft][split] += 1

# Actual achieved counts per fuel type, including the forced-train locations -
# may fall short of target val/test if the eligible pool couldn't supply enough
print("\nActual achieved counts (including forced-train locations, which only ever add to train):")
for ft in STRATIFIED_FUEL_TYPES:
    forced_n = len(sub_threshold_locations & set(interior_strat_all.loc[interior_strat_all["fuel_type"] == ft, "dictionary_id"]))
    achieved_train = already_assigned[ft]["train"] + forced_n
    achieved_val = already_assigned[ft]["val"]
    achieved_test = already_assigned[ft]["test"]
    target = fuel_type_targets[ft]
    shortfall = ""
    if achieved_val < target["val"] or achieved_test < target["test"]:
        shortfall = "  <- short of target, forced-train ate into the eligible pool"
    print(f"  {ft:8s} target train={target['train']:2d} val={target['val']} test={target['test']}  |  "
          f"achieved train={achieved_train:2d} val={achieved_val} test={achieved_test}{shortfall}")

print(f"\n{len(location_split)} locations assigned a split via stratification; {len(sub_threshold_locations)} forced to train separately")

15 interior location(s) forced to train (carry a sub-threshold fuel type): [10000, 10004, 10007, 10010, 10011, 10012, 10013, 10014, 10078, 10088, 10089, 10143, 10144, 10145, 10146]

Per-fuel-type targets (locations, not series - includes forced-train locations in n):
  WIND     n=104 -> train=72, val=16, test=16
  CCGT     n= 38 -> train=26, val=6, test=6
  NPSHYD   n= 13 -> train=9, val=2, test=2
  NUCLEAR  n=  7 -> train=5, val=1, test=1
  OCGT     n= 11 -> train=7, val=2, test=2

1 multi-stratified-fuel-type location group(s) among the eligible pool: [['CCGT', 'OCGT']]

Actual achieved counts (including forced-train locations, which only ever add to train):
  WIND     target train=72 val=16 test=16  |  achieved train=72 val=16 test=16
  CCGT     target train=26 val=6 test=6  |  achieved train=26 val=6 test=6
  NPSHYD   target train= 9 val=2 test=2  |  achieved train= 9 val=2 test=2
  NUCLEAR  target train= 5 val=1 test=1  |  achieved train= 5 val=1 test=1
  OCGT     target train= 7 

In [18]:
def resolve_split(row):
    if row["hull_status"] == "boundary":
        return "train"
    if row["dictionary_id"] in sub_threshold_locations:
        return "train"  # explicit, though location_split.get(..., "train") would already default here
    return location_split.get(row["dictionary_id"], "train")


all_locations = filtered_summary[["dictionary_id", "hull_status"]].drop_duplicates().copy()
all_locations["split"] = all_locations.apply(resolve_split, axis=1)

site_split_assignment = filtered_summary.merge(
    all_locations[["dictionary_id", "split"]], on="dictionary_id", how="left"
)[["dictionary_id", "fuel_type", "hull_status", "split"]]

print(f"{len(all_locations)} locations assigned a split; {len(site_split_assignment)} fuel-series carry that split")
print(all_locations["split"].value_counts())

# Verification: with the forced-train rule, NO sub-threshold series should ever
# end up outside train - unlike the earlier "follows along" resolution, which let
# 3 series leak into val/test. Confirming that's actually zero now.
sub_threshold_rows = site_split_assignment[~site_split_assignment["fuel_type"].isin(STRATIFIED_FUEL_TYPES)]
leaked = sub_threshold_rows[sub_threshold_rows["split"] != "train"]
print(f"\nSub-threshold fuel-series NOT in train (should be 0 under the forced-train rule): {len(leaked)}")
if len(leaked):
    print("*** unexpected - investigate before treating the split as final ***")
    display(leaked)

189 locations assigned a split; 199 fuel-series carry that split
split
train    137
val       26
test      26
Name: count, dtype: int64

Sub-threshold fuel-series NOT in train (should be 0 under the forced-train rule): 0


### Write the lookup table and report the final composition

In [19]:
SPLIT_PATH = REF_DIR / "site_split_assignment.csv"
site_split_assignment.to_csv(SPLIT_PATH, index=False)
print(f"Wrote {len(site_split_assignment)} rows to {SPLIT_PATH}")

# Stop-condition check: every stratified fuel type must have >= 1 site in val and >= 1 in test
print("\nStop-condition check - stratified fuel types must have >= 1 site in val and >= 1 in test:")
stop_triggered = False
for ft in STRATIFIED_FUEL_TYPES:
    ft_rows = site_split_assignment[site_split_assignment["fuel_type"] == ft]
    val_n = (ft_rows["split"] == "val").sum()
    test_n = (ft_rows["split"] == "test").sum()
    ok = val_n >= 1 and test_n >= 1
    print(f"  {ft:8s} val={val_n}, test={test_n}  {'OK' if ok else '*** FAILS FLOOR ***'}")
    if not ok:
        stop_triggered = True

if stop_triggered:
    print("\n*** STOP: at least one stratified fuel type has fewer than 1 site in val or test. "
          "Do not treat this split as final - flag to Simon before proceeding. ***")
else:
    print("\nAll five stratified fuel types clear the floor. No stop-condition triggered.")

# Final composition: split x fuel_type, for Simon to eyeball
final_composition = (
    site_split_assignment.groupby(["fuel_type", "split"]).size().unstack(fill_value=0)
)
for col in ["train", "val", "test"]:
    if col not in final_composition.columns:
        final_composition[col] = 0
final_composition = final_composition[["train", "val", "test"]]
final_composition["total"] = final_composition.sum(axis=1)
final_composition = final_composition.sort_values("total", ascending=False)

print("\nFinal split composition (fuel-series counts):")
final_composition

Wrote 199 rows to ../data/reference/site_split_assignment.csv

Stop-condition check - stratified fuel types must have >= 1 site in val and >= 1 in test:
  WIND     val=16, test=16  OK
  CCGT     val=6, test=6  OK
  NPSHYD   val=2, test=2  OK
  NUCLEAR  val=1, test=1  OK
  OCGT     val=2, test=2  OK

All five stratified fuel types clear the floor. No stop-condition triggered.

Final split composition (fuel-series counts):


split,train,val,test,total
fuel_type,,,,
WIND,80,16,16,112
CCGT,26,6,6,38
NPSHYD,9,2,2,13
OCGT,8,2,2,12
NUCLEAR,6,1,1,8
COAL,5,0,0,5
BIOMASS,4,0,0,4
PS,4,0,0,4
RECIPROCATING,3,0,0,3


**Observations**: both stop-conditions pass — 2yr cutoff excluded 0 series, all five stratified types clear the ≥1-in-val/≥1-in-test floor. Final composition (train/val/test): WIND 80/16/16, CCGT 26/6/6, NPSHYD 9/2/2, OCGT 8/2/2, NUCLEAR 6/1/1, COAL 5/0/0, BIOMASS 4/0/0, PS 4/0/0, RECIPROCATING 3/0/0. Verified zero sub-threshold leakage into val/test.

## Step 8 — Parquet consolidation (derived cache, re-runnable)

Reads every per-fuel-series CSV, writes one consolidated parquet including the `split` column. CSVs and the split lookup remain source of truth.

In [20]:
def consolidate_to_parquet(out_path=INTERIM_DIR / "site_generation_consolidated.parquet"):
    site_files = sorted(SITE_GEN_DIR.glob("*.csv"))
    if not site_files:
        print("No fuel-series CSVs found yet - nothing to consolidate.")
        return None

    frames = [pd.read_csv(f) for f in site_files]
    consolidated = pd.concat(frames, ignore_index=True)

    # Attach site_fuel_id, hull_status, split and name for convenience at load time in the training notebook.
    # Inner join on filtered_summary/site_split_assignment so series excluded by the
    # minimum-history cutoff are dropped from the parquet too, not just from the split table.
    consolidated = consolidated.merge(
        site_split_assignment.merge(
            filtered_summary[["dictionary_id", "fuel_type", "site_fuel_id", "name"]],
            on=["dictionary_id", "fuel_type"],
        ),
        on=["dictionary_id", "fuel_type"], how="inner",
    )
    consolidated.to_parquet(out_path, index=False)
    print(
        f"Wrote {len(consolidated):,} rows across {consolidated['site_fuel_id'].nunique()} fuel-series "
        f"({consolidated['dictionary_id'].nunique()} locations) to {out_path}"
    )
    print(f"File size: {out_path.stat().st_size / 1e6:.1f} MB")
    print(consolidated[["site_fuel_id", "split"]].drop_duplicates()["split"].value_counts())
    return consolidated


consolidated = consolidate_to_parquet()

Wrote 25,078,122 rows across 199 fuel-series (189 locations) to ../data/interim/site_generation_consolidated.parquet
File size: 66.5 MB


split
train    145
val       27
test      27
Name: count, dtype: int64


**Observations**: 25.1M rows, 199 fuel-series, 189 locations, 66.5MB. Row-level split breakdown matches the Step 7 composition table exactly.

## Stopping point

Split written to `data/reference/site_split_assignment.csv`; parquet regenerated with the `split` column. Mixed-fleet-scope tension (acknowledged from the outset) carried into the eventual write-up.